# 🌲 EvaluaPlus - Entrenamiento ResNet50 con Imágenes Propias

Entrena un clasificador de madera **buena / defectuosa** usando tus propias imágenes.

### Antes de empezar:
1. `Entorno de ejecución → Cambiar tipo → T4 GPU`
2. Sube tus imágenes en el **paso 2** (carpetas `good/` y `defect/`)
3. Ajusta los parámetros en el **paso 3** según tus necesidades
4. Ejecuta todas las celdas en orden

## ✅ Paso 1 — Verificar GPU y dependencias

In [ ]:
!nvidia-smi
!pip install -q albumentations torchmetrics scikit-learn matplotlib seaborn
import torch
print(f'\nPyTorch: {torch.__version__} | CUDA disponible: {torch.cuda.is_available()}')

## 📁 Paso 2 — Subir tus imágenes

Sube tus imágenes organizadas así:
```
mis_imagenes/
├── good/        ← imágenes de madera en buen estado
└── defect/      ← imágenes de madera defectuosa
```
Puedes subir un ZIP con esa estructura o usar Google Drive.

In [ ]:
import os, zipfile, shutil
from pathlib import Path
from google.colab import files

# ── OPCIÓN A: Subir un ZIP ────────────────────────────────────────────────────
# El ZIP debe tener carpetas: good/ y defect/ en su raíz
print('Selecciona tu ZIP con las imágenes...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('mis_imagenes')

# Auto-detectar si hay subcarpeta o si las carpetas están en la raíz
base = Path('mis_imagenes')
subdirs = [d for d in base.iterdir() if d.is_dir()]
if len(subdirs) == 1:  # hay una carpeta contenedora
    base = subdirs[0]

# Verificar estructura
for clase in ['good', 'defect']:
    n = len(list((base / clase).glob('*.*'))) if (base / clase).exists() else 0
    estado = '✅' if n > 0 else '❌'
    print(f'{estado} {clase}: {n} imágenes  →  {base / clase}')

RAW_DIR = str(base)  # Guardar ruta para los pasos siguientes
print(f'\nDirectorio base: {RAW_DIR}')

In [ ]:
# ── OPCIÓN B: Google Drive (comenta el bloque A y descomenta esto) ────────────
# from google.colab import drive
# drive.mount('/content/drive')
# RAW_DIR = '/content/drive/MyDrive/evalua_plus/mis_imagenes'  # <- ajusta la ruta

## ⚙️ Paso 3 — Configuración de entrenamiento

Ajusta los parámetros según tus imágenes y recursos disponibles.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#   CONFIGURACIÓN PRINCIPAL  —  modifica lo que necesites
# ══════════════════════════════════════════════════════════════════════

CFG = {
    # ── Imágenes ──────────────────────────────────────────────────────
    'img_size':        224,     # Tamaño de entrada al modelo (224 recomendado para ResNet50)
    'resize_to':       256,     # Redimensionar antes del crop (> img_size; ej: 256 o 288)
    'val_split':       0.2,     # Fracción de datos para validación (0.0–0.5)

    # ── Balance de clases ─────────────────────────────────────────────
    # 'auto'   → calcula pesos inversos al nº de muestras por clase
    # 'manual' → usa los pesos definidos en class_weights_manual
    # 'none'   → sin balance
    'balance_mode':    'auto',
    'class_weights_manual': {'defect': 1.5, 'good': 1.0},  # solo si balance_mode='manual'

    # ── Preprocesamiento / filtros ────────────────────────────────────
    'normalize':       True,    # Normalización ImageNet (recomendado con pesos preentrenados)
    'grayscale':       False,   # Convertir a escala de grises antes de entrenar
    'clahe':           True,    # Mejora de contraste adaptativo (útil en madera)
    'sharpen':         False,   # Nitidez (útil si las fotos son borrosas)

    # ── Aumentación de datos ──────────────────────────────────────────
    'aug_hflip':       True,    # Volteo horizontal
    'aug_vflip':       False,   # Volteo vertical
    'aug_rotate':      15,      # Rotación máxima en grados (0 = desactivado)
    'aug_brightness':  0.2,     # Variación de brillo (0 = desactivado)
    'aug_contrast':    0.2,     # Variación de contraste (0 = desactivado)
    'aug_blur':        True,    # Blur aleatorio (simula fotos movidas)
    'aug_noise':       True,    # Ruido gaussiano
    'aug_elastic':     False,   # Deformación elástica (útil para texturas)
    'aug_cutout':      True,    # Recorta regiones aleatorias (mejora robustez)

    # ── Modelo y entrenamiento ────────────────────────────────────────
    'backbone':        'resnet50',   # resnet50 | resnet34 | efficientnet_b0
    'pretrained':      True,         # Usar pesos de ImageNet
    'freeze_backbone': False,        # Entrenar solo la cabeza final (útil con poco data)
    'epochs':          15,
    'batch_size':      32,
    'lr':              1e-3,
    'lr_scheduler':    'cosine',     # cosine | step | none
    'weight_decay':    1e-4,
    'dropout':         0.3,          # Dropout antes de la capa final (0 = sin dropout)
    'early_stopping':  5,            # Épocas sin mejora antes de parar (0 = desactivado)

    # ── Salida ────────────────────────────────────────────────────────
    'model_name':      'evalua_plus_model.pth',
}

# Clases (orden alfabético = como las carga ImageFolder)
CLASES = ['defect', 'good']

print('✅ Configuración cargada')
for k, v in CFG.items():
    print(f'   {k}: {v}')

## 🔍 Paso 4 — Analizar y balancear el dataset

In [ ]:
import random, shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

raw = Path(RAW_DIR)
extensiones = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}

def listar_imagenes(carpeta):
    return [p for p in carpeta.rglob('*') if p.suffix.lower() in extensiones]

imagenes = {c: listar_imagenes(raw / c) for c in CLASES}

print('📊 Dataset original:')
total = 0
for c, imgs in imagenes.items():
    print(f'   {c}: {len(imgs)} imágenes')
    total += len(imgs)
print(f'   Total: {total}')

# Ratio de desbalance
counts = [len(v) for v in imagenes.values()]
ratio = max(counts) / min(counts) if min(counts) > 0 else float('inf')
print(f'\n⚖️  Ratio de desbalance: {ratio:.2f}x')
if ratio > 3:
    print('   ⚠️  Desbalance alto → se recomienda balance_mode="auto" o aumentar la clase minoritaria')

# ── Dividir train / val manteniendo proporciones ──────────────────────────────
for split in ['train', 'val']:
    for c in CLASES:
        Path(f'dataset/{split}/{c}').mkdir(parents=True, exist_ok=True)

for c in CLASES:
    imgs = imagenes[c]
    random.shuffle(imgs)
    n_val = max(1, int(len(imgs) * CFG['val_split']))
    val_imgs = imgs[:n_val]
    train_imgs = imgs[n_val:]

    for i, img in enumerate(train_imgs):
        shutil.copy(img, f'dataset/train/{c}/{i:05d}{img.suffix}')
    for i, img in enumerate(val_imgs):
        shutil.copy(img, f'dataset/val/{c}/{i:05d}{img.suffix}')

print('\n📂 Dataset dividido:')
for split in ['train', 'val']:
    for c in CLASES:
        n = len(list(Path(f'dataset/{split}/{c}').glob('*')))
        print(f'   {split}/{c}: {n}')

# ── Visualizar muestras ───────────────────────────────────────────────────────
from PIL import Image
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Muestras del dataset', fontsize=14)
for row, c in enumerate(CLASES):
    muestras = list(Path(f'dataset/train/{c}').glob('*'))[:4]
    for col, img_path in enumerate(muestras):
        img = Image.open(img_path).convert('RGB')
        axes[row][col].imshow(img)
        axes[row][col].set_title(c, fontsize=10)
        axes[row][col].axis('off')
plt.tight_layout()
plt.savefig('muestras_dataset.png', dpi=100)
plt.show()
print('✅ División completada')

## 🎨 Paso 5 — Definir preprocesamiento y aumentación

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from PIL import Image
import numpy as np

# ── Construir pipeline de aumentación ────────────────────────────────────────
aug_train = []

# Resize + crop
aug_train.append(A.Resize(CFG['resize_to'], CFG['resize_to']))
aug_train.append(A.RandomCrop(CFG['img_size'], CFG['img_size']))

# Filtros de preprocesamiento
if CFG['clahe']:
    aug_train.append(A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5))
if CFG['sharpen']:
    aug_train.append(A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.4))

# Aumentaciones geométricas
if CFG['aug_hflip']:
    aug_train.append(A.HorizontalFlip(p=0.5))
if CFG['aug_vflip']:
    aug_train.append(A.VerticalFlip(p=0.3))
if CFG['aug_rotate'] > 0:
    aug_train.append(A.Rotate(limit=CFG['aug_rotate'], p=0.5))
if CFG['aug_elastic']:
    aug_train.append(A.ElasticTransform(alpha=1, sigma=20, p=0.3))

# Aumentaciones de color
if CFG['aug_brightness'] > 0 or CFG['aug_contrast'] > 0:
    aug_train.append(A.RandomBrightnessContrast(
        brightness_limit=CFG['aug_brightness'],
        contrast_limit=CFG['aug_contrast'], p=0.5))

# Ruido y blur
if CFG['aug_blur']:
    aug_train.append(A.OneOf([
        A.GaussianBlur(blur_limit=(3, 7)),
        A.MotionBlur(blur_limit=7),
    ], p=0.3))
if CFG['aug_noise']:
    aug_train.append(A.GaussNoise(var_limit=(10.0, 50.0), p=0.3))

# Cutout (simula oclusiones)
if CFG['aug_cutout']:
    aug_train.append(A.CoarseDropout(
        max_holes=8, max_height=CFG['img_size']//8,
        max_width=CFG['img_size']//8, p=0.3))

# Escala de grises (si se activa, duplicamos el canal para mantener 3 canales)
if CFG['grayscale']:
    aug_train.append(A.ToGray(p=1.0))

# Normalización y tensor
if CFG['normalize']:
    aug_train.append(A.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225]))
else:
    aug_train.append(A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]))
aug_train.append(ToTensorV2())

transform_train = A.Compose(aug_train)

# Pipeline de validación (sin aumentación)
aug_val = [A.Resize(CFG['resize_to'], CFG['resize_to']),
           A.CenterCrop(CFG['img_size'], CFG['img_size'])]
if CFG['clahe']:
    aug_val.append(A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0))
if CFG['grayscale']:
    aug_val.append(A.ToGray(p=1.0))
if CFG['normalize']:
    aug_val.append(A.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225]))
else:
    aug_val.append(A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]))
aug_val.append(ToTensorV2())
transform_val = A.Compose(aug_val)

# ── Dataset personalizado con Albumentations ──────────────────────────────────
class WoodDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform
        for idx, clase in enumerate(CLASES):
            folder = Path(root) / clase
            for img_path in folder.glob('*.*'):
                if img_path.suffix.lower() in extensiones:
                    self.samples.append((str(img_path), idx))
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label

train_dataset = WoodDataset('dataset/train', transform=transform_train)
val_dataset   = WoodDataset('dataset/val',   transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'],
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Datasets listos — Train: {len(train_dataset)} | Val: {len(val_dataset)}')
print(f'   Aumentaciones aplicadas: {len(aug_train) - 2} filtros + normalización')

# Preview de aumentación
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Preview aumentación (misma imagen, distintas variaciones)', fontsize=13)
sample_path, sample_label = train_dataset.samples[0]
raw_img = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
for col in range(5):
    aug_result = A.Compose(aug_train[:-2])(image=raw_img)  # sin normalización ni tensor
    axes[0][col].imshow(aug_result['image'])
    axes[0][col].set_title(f'Aug {col+1}', fontsize=9)
    axes[0][col].axis('off')
    axes[1][col].imshow(raw_img)
    axes[1][col].set_title('Original', fontsize=9)
    axes[1][col].axis('off')
plt.tight_layout()
plt.savefig('preview_aumentacion.png', dpi=100)
plt.show()

## 🏗️ Paso 6 — Construir el modelo

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

def construir_modelo(backbone, pretrained, dropout, freeze_backbone):
    if backbone == 'resnet50':
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        if dropout > 0:
            model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, 2))
        else:
            model.fc = nn.Linear(in_features, 2)

    elif backbone == 'resnet34':
        weights = models.ResNet34_Weights.DEFAULT if pretrained else None
        model = models.resnet34(weights=weights)
        in_features = model.fc.in_features
        if dropout > 0:
            model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, 2))
        else:
            model.fc = nn.Linear(in_features, 2)

    elif backbone == 'efficientnet_b0':
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        if dropout > 0:
            model.classifier = nn.Sequential(
                nn.Dropout(p=dropout), nn.Linear(in_features, 2))
        else:
            model.classifier = nn.Sequential(nn.Linear(in_features, 2))
    else:
        raise ValueError(f'Backbone no soportado: {backbone}')

    if freeze_backbone:
        for name, param in model.named_parameters():
            if 'fc' not in name and 'classifier' not in name:
                param.requires_grad = False
        print(f'🔒 Backbone congelado → solo se entrena la cabeza final')

    return model.to(device)

model = construir_modelo(CFG['backbone'], CFG['pretrained'],
                         CFG['dropout'], CFG['freeze_backbone'])

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'✅ Modelo {CFG["backbone"]} creado')
print(f'   Parámetros entrenables: {trainable:,} / {total_params:,}')

## 🚀 Paso 7 — Entrenamiento

In [ ]:
import torch.nn as nn
from torchmetrics.classification import BinaryF1Score, BinaryAUROC

# ── Balance de clases ─────────────────────────────────────────────────────────
if CFG['balance_mode'] == 'auto':
    counts_train = [len(list(Path(f'dataset/train/{c}').glob('*'))) for c in CLASES]
    total_train = sum(counts_train)
    weights = torch.tensor(
        [total_train / (len(CLASES) * c) for c in counts_train], dtype=torch.float
    ).to(device)
    print(f'⚖️  Pesos automáticos: {dict(zip(CLASES, weights.cpu().tolist()))}')
elif CFG['balance_mode'] == 'manual':
    weights = torch.tensor(
        [CFG['class_weights_manual'][c] for c in CLASES], dtype=torch.float
    ).to(device)
    print(f'⚖️  Pesos manuales: {CFG["class_weights_manual"]}')
else:
    weights = None
    print('⚖️  Sin balance de clases')

criterion = nn.CrossEntropyLoss(weight=weights)

# ── Optimizador ───────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG['lr'], weight_decay=CFG['weight_decay'])

# ── Scheduler ─────────────────────────────────────────────────────────────────
if CFG['lr_scheduler'] == 'cosine':
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG['epochs'])
elif CFG['lr_scheduler'] == 'step':
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
else:
    scheduler = None

# ── Métricas ──────────────────────────────────────────────────────────────────
f1_metric  = BinaryF1Score().to(device)
auc_metric = BinaryAUROC().to(device)

# ── Bucle de entrenamiento ────────────────────────────────────────────────────
historia = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
            'val_f1': [], 'val_auc': []}

mejor_val_acc = 0
mejor_val_f1  = 0
epocas_sin_mejora = 0

print(f'\n🚀 Entrenando {CFG["backbone"]} por {CFG["epochs"]} épocas...\n')

for epoch in range(CFG['epochs']):
    # ── TRAIN ──────────────────────────────────────────────────────────
    model.train()
    train_loss = train_correct = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    # ── VAL ────────────────────────────────────────────────────────────
    model.eval()
    val_loss = val_correct = 0
    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()
            probs = torch.softmax(outputs, dim=1)[:, 1]  # prob clase 'good'
            all_preds.append(outputs.argmax(1))
            all_probs.append(probs)
            all_labels.append(labels)

    all_preds  = torch.cat(all_preds)
    all_probs  = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)

    train_acc = train_correct / len(train_dataset) * 100
    val_acc   = val_correct   / len(val_dataset)   * 100
    t_loss    = train_loss / len(train_loader)
    v_loss    = val_loss   / len(val_loader)
    val_f1    = f1_metric(all_preds, all_labels).item()
    val_auc   = auc_metric(all_probs, all_labels).item()

    historia['train_loss'].append(t_loss)
    historia['val_loss'].append(v_loss)
    historia['train_acc'].append(train_acc)
    historia['val_acc'].append(val_acc)
    historia['val_f1'].append(val_f1)
    historia['val_auc'].append(val_auc)

    lr_actual = optimizer.param_groups[0]['lr']
    print(f'Ep {epoch+1:02d}/{CFG["epochs"]} │ '
          f'Loss {t_loss:.4f}/{v_loss:.4f} │ '
          f'Acc {train_acc:.1f}%/{val_acc:.1f}% │ '
          f'F1 {val_f1:.3f} │ AUC {val_auc:.3f} │ lr {lr_actual:.2e}')

    # Guardar mejor modelo (por F1 en validación)
    if val_f1 > mejor_val_f1:
        mejor_val_f1  = val_f1
        mejor_val_acc = val_acc
        torch.save(model.state_dict(), 'mejor_modelo.pth')
        epocas_sin_mejora = 0
        print(f'           ✅ Mejor modelo guardado (F1={val_f1:.3f})')
    else:
        epocas_sin_mejora += 1

    if scheduler:
        scheduler.step()

    # Early stopping
    if CFG['early_stopping'] > 0 and epocas_sin_mejora >= CFG['early_stopping']:
        print(f'\n⏹️  Early stopping en época {epoch+1}')
        break

print(f'\n🎉 Entrenamiento completado → Mejor F1: {mejor_val_f1:.3f} | Acc: {mejor_val_acc:.1f}%')

## 📊 Paso 8 — Curvas de entrenamiento y matriz de confusión

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ── Cargar mejor modelo para evaluación ──────────────────────────────────────
model.load_state_dict(torch.load('mejor_modelo.pth', map_location=device))
model.eval()

# ── Curvas ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_range = range(1, len(historia['train_acc']) + 1)

axes[0].plot(epochs_range, historia['train_acc'], label='Train')
axes[0].plot(epochs_range, historia['val_acc'],   label='Val')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].set_xlabel('Época')

axes[1].plot(epochs_range, historia['train_loss'], label='Train')
axes[1].plot(epochs_range, historia['val_loss'],   label='Val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].set_xlabel('Época')

axes[2].plot(epochs_range, historia['val_f1'],  label='F1')
axes[2].plot(epochs_range, historia['val_auc'], label='AUC')
axes[2].set_title('Val F1 & AUC'); axes[2].legend(); axes[2].set_xlabel('Época')

plt.tight_layout()
plt.savefig('curvas_entrenamiento.png', dpi=120)
plt.show()

# ── Matriz de confusión ───────────────────────────────────────────────────────
y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        y_pred.extend(outputs.argmax(1).cpu().numpy())
        y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASES, yticklabels=CLASES)
plt.title('Matriz de Confusión (validación)')
plt.ylabel('Real'); plt.xlabel('Predicho')
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=120)
plt.show()

print('\n📋 Reporte por clase:')
print(classification_report(y_true, y_pred, target_names=CLASES))

## 💾 Paso 9 — Guardar modelo con metadata completa

In [ ]:
# Cargar el mejor modelo guardado
model.load_state_dict(torch.load('mejor_modelo.pth', map_location='cpu'))
model.eval()

# Guardar con toda la metadata necesaria para inference en el backend
torch.save({
    'model_state_dict': model.state_dict(),
    'clases':           CLASES,
    'arquitectura':     CFG['backbone'],
    'input_size':       (CFG['img_size'], CFG['img_size']),
    'normalize':        CFG['normalize'],
    'clahe':            CFG['clahe'],
    'grayscale':        CFG['grayscale'],
    'mejor_val_f1':     round(mejor_val_f1, 4),
    'mejor_val_acc':    round(mejor_val_acc, 2),
    'num_train':        len(train_dataset),
    'num_val':          len(val_dataset),
}, CFG['model_name'])

print(f'✅ Modelo guardado: {CFG["model_name"]}')
print(f'   Arquitectura : {CFG["backbone"]}')
print(f'   Clases       : {CLASES}')
print(f'   Mejor F1     : {mejor_val_f1:.4f}')
print(f'   Mejor Acc    : {mejor_val_acc:.2f}%')

## ⬇️ Paso 10 — Descargar modelo y gráficas

In [ ]:
from google.colab import files

files.download(CFG['model_name'])
files.download('curvas_entrenamiento.png')
files.download('matriz_confusion.png')
files.download('muestras_dataset.png')

print('✅ Archivos descargados. Coloca el .pth en: ai/model/evalua_plus_model.pth')